# SFT: messy text → normalized JSON extraction

This notebook is a **thin demo**: every step calls into the tested `geap_tuning`
package. It mirrors [`examples/run_sft_extraction.py`](../examples/run_sft_extraction.py).

Our other SFT demos are all **classification** (support-intent, banking77,
oral-disease images). This one is **generative structured output**: rewrite a
messy order line into a strict five-key JSON object. Plain extraction saturates a
modern base (it scored a perfect `accuracy` in a live run), so this teaches a
**house normalization standard the base cannot guess** — P-code priorities
(`urgent`→`P0`), city-abbreviation expansion (`NYC`→`New York`), `ord-` prefix
stripping, spelled-out quantities. A **pilot gate** scores the untuned base first
and only tunes if it is below the saturation ceiling (real headroom). The
`per_field` breakdown shows exactly where the base fails (arbitrary P-codes) and
where it already succeeds (it knows `NYC`→`New York`).

> **Requires live GCP and incurs tuning cost** (one SFT job). Have a real `.env`
> and `gcloud auth` in place.

In [ ]:
from geap_tuning.config import genai_client, load_config

BASE_MODEL = "gemini-2.5-flash"
SAT_CEILING = 0.85  # base field-exact-match accuracy must be below this to have headroom
cfg = load_config()
client = genai_client(cfg)
cfg

## 1. Build the dataset and stage it to GCS

The bank is generated deterministically (correct-by-construction): each messy
input line uses raw tokens (`ord-a1234`, `NYC`, `urgent`, `a dozen`) and the gold
JSON carries the **normalized** values (`A1234`, `New York`, `P0`, `12`). The
`SYSTEM_INSTRUCTION` signals that an internal standard applies but does **not**
spell out the arbitrary mapping — that is what SFT learns from the labels.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.sft.extraction import (
    EXTRACTION_EXAMPLES,
    SYSTEM_INSTRUCTION,
    build_extraction_dataset,
    build_records,
    split_dataset,
)

GCS_PREFIX = "sft_json_extraction_v2"
paths = build_extraction_dataset("../datasets/sft_json_extraction_v2")
train_uri = upload_file(paths["train"], f"{cfg.bucket}/{GCS_PREFIX}/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/{GCS_PREFIX}/val.jsonl")

_, _, test = split_dataset(EXTRACTION_EXAMPLES)
test_records = build_records(test)
print(f"{len(EXTRACTION_EXAMPLES)} examples, {len(test_records)} held out")
print(SYSTEM_INSTRUCTION)

## 2. Pilot gate — score the untuned base (the "before")

`run_eval` parses each reply as JSON (stripping fences / prose) and scores micro
field-exact-match `accuracy`, `json_validity`, whole-object `exact_match`, and a
`per_field` breakdown. The gate proceeds only if base `accuracy < SAT_CEILING` —
otherwise the base already knows our convention and there is nothing to teach.

In [ ]:
from geap_tuning.inference import generate
from geap_tuning.sft.extraction_eval import run_eval

base = run_eval(
    test_records,
    predict_fn=lambda t: generate(client, BASE_MODEL, t, system_instruction=SYSTEM_INSTRUCTION),
)
print(
    f"BASE accuracy={base['accuracy']:.3f} exact_match={base['exact_match']:.3f} "
    f"json_validity={base['json_validity']:.3f}"
)
print(f"per_field: {base['per_field']}")
if base["accuracy"] < SAT_CEILING:
    print("Pilot gate PASSED — the base does not know our convention; headroom confirmed.")
else:
    print("WARNING: base already applies the convention (no headroom) — SFT may not show a lift.")

## 3. Launch the SFT job and wait

Reuse an existing job with the same display name if one exists (cost control);
otherwise launch a fresh supervised job on the same `client.tunings.tune(...)`
call with `method="SUPERVISED"`.

In [ ]:
from geap_tuning.jobs import find_tuning_job_by_display_name, tuned_endpoint, wait_for_tuning_job
from geap_tuning.sft.tune import launch_sft_job

DISPLAY_NAME = "geap-sft-json-extraction-v2"

job = find_tuning_job_by_display_name(client, DISPLAY_NAME)
if job is None:
    job = launch_sft_job(
        client,
        train_uri=train_uri,
        val_uri=val_uri,
        display_name=DISPLAY_NAME,
        base_model=BASE_MODEL,
        labels=cfg.labels,
    )
job = wait_for_tuning_job(client, job.name)
endpoint = tuned_endpoint(job)
endpoint

## 4. Score the tuned endpoint (the "after") and report the lift

In [ ]:
tuned = run_eval(
    test_records,
    predict_fn=lambda t: generate(client, endpoint, t, system_instruction=SYSTEM_INSTRUCTION),
)
print(
    f"TUNED accuracy={tuned['accuracy']:.3f} exact_match={tuned['exact_match']:.3f} "
    f"json_validity={tuned['json_validity']:.3f}"
)
print(f"per_field: {tuned['per_field']}")
print(
    f"LIFT accuracy +{tuned['accuracy'] - base['accuracy']:.3f}; "
    f"exact_match +{tuned['exact_match'] - base['exact_match']:.3f}; "
    f"json_validity {base['json_validity']:.3f}->{tuned['json_validity']:.3f}"
)